# Genotype Host Table

Creating a table that, per genotype, shows what hosts have been infected with that genotype.

## Housekeeping

In [1]:
import os
import pandas as pd
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Dates
start_date = "11-01-2021"
end_date = "06-13-2025"
date_range = start_date + "--" + end_date
# genotypes = ["D1.1", "B3.2", "B3.6", "B3.13", "A3", "A2", "C2.1"]

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"

combined_files_old = home + "Combinations/GISAID_Andersen_NCBI_Virus/11-01-2021--04-14-2025_all_genotypes_Antarctica_North_America_South_America/"
combined_files_new = home + "Combinations/GISAID_Andersen_NCBI_Virus/04-14-2025--6-13-2025_all_genotypes_Antarctica_North_America_South_America/"
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
# references = "C:/Users/maksi/Documents/Statistics/projects/Avian_Flu/references/"

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")
genotypes_df = pd.read_excel("genotype_key.xlsx")
animals_ref = pd.read_csv("animals_ref.csv")

# genotypes = list(genotypes_df["Genotype"])

In [2]:
# Function to prepare dataframes
def fasta_df_og(file_name, state_ref):

    fasta = pd.DataFrame()
    headers = []
    isolate_ids = []
    isolate_names = []
    subtypes = []
    # segments = []
    collection_dates = []
    sequences = []
    host_types = []
    species = []
    identifiers = []
    genotypes = []
    with open(file_name) as f:
        lines = f.readlines()
        for num, line in enumerate(lines):
            # print(line)
            if line[0] == ">": # If it's a header
                if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
                    header = line[1:].strip() # Remove the ">"
                    # print(header)
                    split_header = header.split("|")
                    if len(header.split("|")) > 6:
                        identifier = header.split("|")[0]
                        identifiers.append(identifier)
                        split_first_header = header.split("|")[1].split("/")
                    # elif len(header.split("|")) == 6:
                    #     identifier = header.split("|")[0]
                    #     identifiers.append(identifier)
                    #     split_first_header = header.split("|")[1].split("/")
                    else:
                        identifiers.append("unknown")
                        split_first_header = split_header[-5].split("/")
                    
                    # print(split_first_header)
                    # print(split_header)
                    headers.append(header) 

                    if len(header.split("|")) > 6:
                        # print(header)
                        isolate_ids.append(split_first_header[3])
                        isolate_names.append("/".join(split_first_header)) # We'll need to extract data from this too
                        subtypes.append(split_header[2])  # Get only H5N1
                        genotypes.append(split_header[-1])
                        # segments.append(split_header[].split("_")[-1])
                        host_types.append(split_header[-2])
                        species.append(split_first_header[1])
                        # if split_header[4] == "2024-01-01":
                        #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                        # elif split_header[4] == "2025-01-01":
                        #     collection_dates.append("2025")
                        # else: 
                        collection_dates.append(split_header[-3].split("_")[-1])
                        if num < len(lines): # If we're not at the last line
                            # for i, l in enumerate(lines[num + 1:]):
                            i = num
                            sequence = ""
                            # print(lines[i])
                            # print(lines[i + 1])
                            while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                                sequence = sequence + lines[i + 1].strip()
                                i += 1
                            sequences.append(sequence) # Add next line to sequences
                    else:

                        isolate_ids.append(split_first_header[3])
                        isolate_names.append("/".join(split_first_header)) # We'll need to extract data from this too
                        subtypes.append(split_header[-4])  # Get only H5N1
                        genotypes.append(split_header[-1])
                        # segments.append(split_header[].split("_")[-1])
                        host_types.append(split_header[-2])
                        species.append(split_first_header[1])
                        # if split_header[4] == "2024-01-01":
                        #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                        # elif split_header[4] == "2025-01-01":
                        #     collection_dates.append("2025")
                        # else: 
                        collection_dates.append(split_header[-3].split("_")[-1])
                        if num < len(lines): # If we're not at the last line
                            # for i, l in enumerate(lines[num + 1:]):
                            i = num
                            sequence = ""
                            # print(lines[i])
                            # print(lines[i + 1])
                            while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                                sequence = sequence + lines[i + 1].strip()
                                i += 1
                            sequences.append(sequence) # Add next line to sequences
        f.close()

    # Create columns for data frame 
    fasta["Header"] = headers
    fasta["Isolate_Id"] = isolate_ids
    fasta["Isolate_Name"] = isolate_names
    fasta["Subtype"] = subtypes
    # fasta["Segment"] = segments
    # Geo_Location is more complicated
    fasta["Geo_Location"] = fasta["Header"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    fasta["Date Collected"] = collection_dates
    fasta["Species"] = species
    fasta["Host_Type"] = host_types
    fasta["Genotype"] = genotypes
    fasta["Sequence"] = sequences
    if len(identifiers) == len(fasta):
        fasta["Identifier"] = identifiers
    
    return fasta

In [3]:
original_fasta_dfs = {}

for dirpath, dirs, files in os.walk(combined_files_old):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            # try:
            fasta_file = fasta_df_og(file_name, states_ref)
            # except:
                
            original_fasta_dfs[file_name] = fasta_file
            # print(fasta_file)
            # break 
    break 

for dirpath, dirs, files in os.walk(combined_files_new):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            fasta_file = fasta_df_og(file_name, states_ref)
            original_fasta_dfs[file_name] = fasta_file
            print(fasta_file)
            # break 
    break 

                                              Header Isolate_Id  \
0  EPI_ISL_19873848|A/wood_duck/Ohio/23OS0170/202...   23OS0170   

                     Isolate_Name Subtype Geo_Location Date Collected  \
0  A/wood_duck/Ohio/23OS0170/2023    H5N1       USA-OH     2023-10-29   

     Species Host_Type Genotype  \
0  wood_duck     avian       A2   

                                            Sequence        Identifier  
0  atggagaacatagtactacttcttgcaatagttagccttgttaaaa...  EPI_ISL_19873848  
                                              Header Isolate_Id  \
0  EPI_ISL_19873848|A/wood_duck/Ohio/23OS0170/202...   23OS0170   

                     Isolate_Name Subtype Geo_Location Date Collected  \
0  A/wood_duck/Ohio/23OS0170/2023    H5N1       USA-OH     2023-10-29   

     Species Host_Type Genotype  \
0  wood_duck     avian       A2   

                                            Sequence        Identifier  
0  atgagtcttctaaccgaggtcgaaacgtacgttctctctatcgtcc...  EPI_ISL_19873848  
  

In [4]:
print(len(original_fasta_dfs))

# Concatenate similar genotypes

concat_fasta_dfs = {}
seen = []
for key in original_fasta_dfs:
    # print(key.split("/")[-1])
    for key1 in original_fasta_dfs: # Look in the same list
        
        if key.split("/")[-1].split("_")[:2] == key1.split("/")[-1].split("_")[:2] and key1 not in seen:
            print("_".join(key.split("/")[-1].split("_")[:2]))
            concat_fasta_dfs[key.split("/")[-1]] = pd.concat([original_fasta_dfs[key], original_fasta_dfs[key1]])
            seen.append(key1)

print(concat_fasta_dfs["B3.2_NA_combined_04-14-2025.fasta"])

672
A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_HA
A2_MP
A2_MP
A2_NA
A2_NA
A2_NP
A2_NP
A2_NS
A2_NS
A2_PA
A2_PA
A2_PB1
A2_PB1
A2_PB2
A2_PB2
A3_HA
A3_HA
A3_MP
A3_MP
A3_NA
A3_NA
A3_NP
A3_NP
A3_NS
A3_NS
A3_PA
A3_PA
A3_PB1
A3_PB1
A3_PB2
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_HA
B3.13_MP
B3.13_MP
B3.13_NA

In [5]:
print(len(concat_fasta_dfs)/8)

75.0


In [6]:
genotypes = set()
for key in concat_fasta_dfs:
    genotype = key.split("_")[0] # + "_HA" # Make sure we only get one of the 8
    # print(genotype)

    genotypes.add(genotype)

genotypes = sorted(list(genotypes))

print(genotypes)

['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'B1.1', 'B1.2', 'B1.3', 'B2.1', 'B2.2', 'B3.1', 'B3.10', 'B3.11', 'B3.12', 'B3.13', 'B3.2', 'B3.3', 'B3.4', 'B3.5', 'B3.6', 'B3.7', 'B3.8', 'B3.9', 'B4.1', 'B5.1', 'C1.1', 'C2.1', 'C3.1', 'D1.1', 'D1.2', 'D1.3', 'Minor01', 'Minor04', 'Minor07', 'Minor08', 'Minor09', 'Minor100', 'Minor103', 'Minor105', 'Minor11', 'Minor12', 'Minor13', 'Minor14', 'Minor15', 'Minor17', 'Minor19', 'Minor28', 'Minor33', 'Minor34', 'Minor35', 'Minor38', 'Minor45', 'Minor50', 'Minor51', 'Minor55', 'Minor57', 'Minor58', 'Minor60', 'Minor61', 'Minor62', 'Minor63', 'Minor66', 'Minor70', 'Minor73', 'Minor76', 'Minor77', 'Minor79', 'Minor81', 'Minor83', 'Minor86', 'Minor89', 'Minor91', 'Minor94', 'Minor97']


In [7]:
host_table = pd.DataFrame(columns=genotypes)

print(animals_ref)

all_animals = []
for col in animals_ref.columns:
    # col = col.split("_")[0]
    if col != "avian":
        for animal in animals_ref[col].values:
            if animal == animal: # If not nan
                all_animals.append(animal)

host_table["animals"] = all_animals
host_table.index = host_table["animals"]

print(host_table)
print(all_animals)
print(len(all_animals))

                   avian               cattle        feline   other_mammal  \
0       great_horned_owl            dairy_cow           cat     deer mouse   
1           common_raven               cattle  domestic_cat    house_mouse   
2          cooper's_hawk  cattle milk product     feral_cat          skunk   
3           coopers_hawk          bovine_milk        feline  striped_skunk   
4                peafowl              bovine   domestic-cat     norway rat   
..                   ...                  ...           ...            ...   
458         embden_goose                  NaN           NaN            NaN   
459  american_buff_goose                  NaN           NaN            NaN   
460   indian_runner_duck                  NaN           NaN            NaN   
461               fisher                  NaN           NaN            NaN   
462      cooper's_s_hawk                  NaN           NaN            NaN   

          human         other  
0    washington         mixed  

In [8]:
genotype_animals = {}

for genotype in genotypes:
    for key in concat_fasta_dfs:
        if genotype in key:
            df = concat_fasta_dfs[key]
            df_corrected = df[df["Genotype"] == genotype.split("_")[0]]
            genotype_animals[genotype.split("_")[0]] = list(df_corrected["Species"].apply(lambda x: x.lower()).values)
            print(df_corrected)

print(genotype_animals)
        # break

                                                Header     Isolate_Id  \
0    A/eared_grebe/Wyoming/22-017947-002/2022|H5N1|...  22-017947-002   
1    A/eared_grebe/Wyoming/22-017947-001/2022|H5N1|...  22-017947-001   
2    A/double_crested_cormorant/Wisconsin/22-030264...  22-030264-001   
3    A/gadwall/Tennessee/22-024871-012/2022|H5N1|20...  22-024871-012   
4    A/gadwall/Tennessee/22-024871-011/2022|H5N1|20...  22-024871-011   
..                                                 ...            ...   
592  A/turkey/Missouri/22-007087-002/2022|H5N1|2022...  22-007087-002   
593  A/chicken/Delaware/22-006945-001/2022|H5N1|202...  22-006945-001   
594  A/turkey/Missouri/22-006944-002/2022|H5N1|2022...  22-006944-002   
595  A/chicken/Maryland/22-006948-001/2022|H5N1|202...  22-006948-001   
596  A/chicken/Delaware/22-006945-002/2022|H5N1|202...  22-006945-002   

                                          Isolate_Name Subtype Geo_Location  \
0             A/eared_grebe/Wyoming/22-01794

In [9]:
for animal in host_table["animals"].values:
    for genotype in genotypes:
        if genotype in genotype_animals:
            animals_list = genotype_animals[genotype]
            if animal in animals_list:
                host_table.at[animal, genotype] = True

In [10]:
display(host_table)

,A1,A2,A3,A4,A5,A6,B1.1,B1.2,B1.3,B2.1,...,Minor77,Minor79,Minor81,Minor83,Minor86,Minor89,Minor91,Minor94,Minor97,animals
animals,,,,,,,,,,,,,,,,,,,,,
dairy_cow,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dairy_cow
cattle,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cattle
cattle milk product,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cattle milk product
bovine_milk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bovine_milk
bovine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bovine
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
environment,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,environment
multispecies,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,multispecies
unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unknown


In [ ]:
os.chdir(home)

# host_table.to_csv("host_table_no_avian.csv")